<a href="https://colab.research.google.com/github/Aqifcodes/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aqifcodes/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone https://github.com/Aqifcodes/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 127 (delta 39), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 1.87 MiB | 14.82 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/flyrank-ml-internship


This task is framed as a Ranking / Binary Classification problem. The goal is to predict which web pages are experiencing a performance drop (is_declining_label = 1) and rank them so content operations teams can prioritize which pages to review first.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
import pandas as pd
import numpy as np

# Load starter data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create target label from trend_direction
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Total rows (pages): {len(df)}")
print("\nClass distribution for is_declining_label:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

Total rows (pages): 30000

Class distribution for is_declining_label:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is is_declining_label (1 = declining, 0 = non-declining). It is a defined proxy rule derived from historical impression trajectories (trend_direction == 'down'). It uses observable historical signals without including forward-looking leakage metrics.

In [ ]:
# Verify relationship between trend_direction and target label
target_check = pd.crosstab(df["trend_direction"], df["is_declining_label"])
print("Target Distribution across Trend Directions:")
print(target_check)

Target Distribution across Trend Directions:
is_declining_label     0      1
trend_direction                
down                   0  16262
flat                1152      0
new                 2236      0
stable              5962      0
up                  4388      0


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary success metric is Precision@50. Because operations teams have limited time and inspect only the top 50 flagged pages each week, maximizing the fraction of actual declining pages in those top 50 prevents wasted effort on false alarms.

In [ ]:
# Compute baseline Precision@50 on dataset
y_true = df["is_declining_label"].values
random_p50 = y_true[:50].mean()

print(f"Overall Declining Rate in Dataset: {y_true.mean():.3f}")
print(f"Random Selection Precision@50 Baseline: {random_p50:.3f}")

Overall Declining Rate in Dataset: 0.542
Random Selection Precision@50 Baseline: 0.680


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one unique web page (URL) per aggregated observation period. Each row pairs 90-day search traffic performance metrics (impressions, CTR, average position) with page metadata (word count, content age).

In [ ]:
# Inspect unit of analysis schema using existing dataset columns
unit_sample = df[["impressions_90d", "avg_position", "ctr", "content_age_days", "days_since_last_update", "is_declining_label"]].head(3)
print("Unit of Analysis DataFrame Sample (1 row = 1 unique page):")
unit_sample

Unit of Analysis DataFrame Sample (1 row = 1 unique page):


,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,is_declining_label
0,3803,10.6,0.76,187,20,1
1,15320,20.3,0.05,445,25,1
2,12581,36.5,0.09,141,20,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule relies on rigid cutoffs (e.g., content_age > 180 days) that miss subtle interactions between position drops, CTR collapse, and traffic scale. Machine learning models combine multiple weak, non-linear signals across all columns simultaneously, resulting in significantly higher Precision@50 than simple hand-coded rules.

In [ ]:
# Calculate hand rule Precision@50 vs dataset average
hand_rule = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).astype(int) * df["impressions_90d"]

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(f"Hand Rule Precision@50: {precision_at_k(hand_rule, df['is_declining_label'], 50):.3f}")

Hand Rule Precision@50: 0.680


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.